# TSE Prompt Preview: With vs. Without Question Text

Shows how a TSE ICL prompt looks under the two ablation conditions:
- **Condition A** (`use_label_desc=0`): model sees only labeled TS examples — no question, no option names
- **Condition B** (`use_label_desc=1`): question + option legend prepended to the prompt

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'ts-icl') if 'icl_results' in os.getcwd() else os.getcwd())
# Make sure we're running from the project root
if os.path.basename(os.getcwd()) == 'icl_results':
    os.chdir('..')
print('Working dir:', os.getcwd())

Working dir: /home/aviramom/projects/ts-icl


In [2]:
import argparse
import json

from data_provider.dataset_tse import TimeSeriesExamDataset
from data_provider.icl_dataset import ICLUCRDataset

TSE_DATA = "qa_dataset_augmented.json"
TID = 1  # change to any available tid

# Show available tids
with open(TSE_DATA) as f:
    all_entries = json.load(f)
available_tids = sorted({e['tid'] for e in all_entries})
print(f"Available tids: {available_tids}")

Available tids: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 95, 96, 97, 98, 99, 100, 101, 102]


In [3]:
# Load the dataset for both splits
train_ds = TimeSeriesExamDataset(TSE_DATA, tid=TID, split="train", test_fraction=0.3, seed=42)
test_ds  = TimeSeriesExamDataset(TSE_DATA, tid=TID, split="test",  test_fraction=0.3, seed=42)

print(f"tid={TID}")
print(f"Question  : {next(e['question'] for e in all_entries if e['tid'] == TID)}")
print(f"Options   : {train_ds.option_texts}")
print(f"Labels    : {train_ds.label_names}")
print(f"is_two_series: {train_ds.is_two_series}")
print(f"Train size: {len(train_ds)}, Test size: {len(test_ds)}")
print()
print("=== Description (use_label_desc=1 injects this) ===")
print(train_ds.desc)

tid=1
Question  : What is the type of the trend of the given time series?
Options   : ['Linear', 'Exponential', 'No Trend']
Labels    : ['A', 'B', 'C']
is_two_series: False
Train size: 21, Test size: 9

=== Description (use_label_desc=1 injects this) ===
Question: What is the type of the trend of the given time series?

Options:
A) Linear
B) Exponential
C) No Trend


In [4]:
def make_args(use_label_desc: int) -> argparse.Namespace:
    return argparse.Namespace(
        picking_strategy="first",
        num_shots=1,
        random_seed=42,
        use_label_desc=use_label_desc,
        tse_data_path=TSE_DATA,
        tse_test_fraction=0.3,
        input_mode="combined",
        desc_dir="ucr_descriptions",
    )

task_id = f"icl_tse_{TID}"

icl_no_desc   = ICLUCRDataset.from_ucr_dataset(train_ds, test_ds, task_id, make_args(0))
icl_with_desc = ICLUCRDataset.from_ucr_dataset(train_ds, test_ds, task_id, make_args(1))

print(f"Samples in test set: {len(icl_no_desc)}")

Samples in test set: 9


In [5]:
SAMPLE_IDX = 0  # which test sample to display

prompt_no_desc   = icl_no_desc[SAMPLE_IDX]["input_text"]
prompt_with_desc = icl_with_desc[SAMPLE_IDX]["input_text"]
true_label       = icl_no_desc[SAMPLE_IDX]["output_text"]

print(f"True label: {true_label}")

True label: A


## Condition A — No question text (`use_label_desc=0`)

In [6]:
# Truncate TS values for readability
def truncate_ts(prompt: str, max_values: int = 8) -> str:
    import re
    def shorten(m):
        vals = [v.strip() for v in m.group(1).split(',')]
        if len(vals) > max_values:
            shown = ', '.join(vals[:max_values])
            return f'[{shown}, ... ({len(vals)} values)]'
        return m.group(0)
    return re.sub(r'\[([^\]]+)\]', shorten, prompt)

print(truncate_ts(prompt_no_desc))

Time Series Classification.


--- EXAMPLES ---

Example 1 Time Series: [0.0491, -0.0171, 0.1237, 0.2489, 0.5246, 0.7387, 0.2755, -0.1334, ... (256 values)]
Label: A

Example 2 Time Series: [1.3657, 0.4069, 2.0581, 1.0753, 0.9721, 2.7193, 0.9586, 1.6550, ... (256 values)]
Label: B

Example 3 Time Series: [7.4760, 7.5438, 7.4944, 7.6086, 7.5968, 7.5744, 7.7040, 7.6933, ... (256 values)]
Label: C

--- TARGET ---
New Time Series: [-1.0386, 0.5888, -0.5573, 0.9470, 0.7866, 0.0591, 0.6072, -2.1789, ... (256 values)]
Return ONLY the label as one of: [A, B, C] without any explanation



## Condition B — With question text (`use_label_desc=1`)

In [7]:
print(truncate_ts(prompt_with_desc))

Time Series Classification.
Question: What is the type of the trend of the given time series?

Options:
A) Linear
B) Exponential
C) No Trend

--- EXAMPLES ---

Example 1 Time Series: [0.0491, -0.0171, 0.1237, 0.2489, 0.5246, 0.7387, 0.2755, -0.1334, ... (256 values)]
Label: A

Example 2 Time Series: [1.3657, 0.4069, 2.0581, 1.0753, 0.9721, 2.7193, 0.9586, 1.6550, ... (256 values)]
Label: B

Example 3 Time Series: [7.4760, 7.5438, 7.4944, 7.6086, 7.5968, 7.5744, 7.7040, 7.6933, ... (256 values)]
Label: C

--- TARGET ---
New Time Series: [-1.0386, 0.5888, -0.5573, 0.9470, 0.7866, 0.0591, 0.6072, -2.1789, ... (256 values)]
Return ONLY the label as one of: [A, B, C] without any explanation



## Diff — what the description adds

In [8]:
import difflib

a_lines = truncate_ts(prompt_no_desc).splitlines(keepends=True)
b_lines = truncate_ts(prompt_with_desc).splitlines(keepends=True)

diff = difflib.unified_diff(a_lines, b_lines, fromfile="no_desc", tofile="with_desc", lineterm="")
print("".join(diff))

--- no_desc+++ with_desc@@ -1,5 +1,10 @@ Time Series Classification.
+Question: What is the type of the trend of the given time series?
 
+Options:
+A) Linear
+B) Exponential
+C) No Trend
 
 --- EXAMPLES ---
 



## All test samples for this template

In [9]:
print(f"{'Idx':>3}  {'True label':>10}  {'Options':>30}")
print("-" * 50)
for i in range(len(icl_no_desc)):
    s = icl_no_desc[i]
    print(f"{i:>3}  {s['output_text']:>10}  {str(s['options']):>30}")

Idx  True label                         Options
--------------------------------------------------
  0           A                 ['A', 'B', 'C']
  1           A                 ['A', 'B', 'C']
  2           A                 ['A', 'B', 'C']
  3           B                 ['A', 'B', 'C']
  4           B                 ['A', 'B', 'C']
  5           B                 ['A', 'B', 'C']
  6           C                 ['A', 'B', 'C']
  7           C                 ['A', 'B', 'C']
  8           C                 ['A', 'B', 'C']
